# 这是THU-COAI

In [1]:
import torch
from transformers.models.bert import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained('thu-coai/roberta-base-cold')
model = BertForSequenceClassification.from_pretrained('thu-coai/roberta-base-cold')
model.eval()

texts = ['你就是个傻逼！','黑人很多都好吃懒做，偷奸耍滑！','男女平等，黑人也很优秀。']

model_input = tokenizer(texts,return_tensors="pt",padding=True)
model_output = model(**model_input, return_dict=False)
prediction = torch.argmax(model_output[0].cpu(), dim=-1)
# prediction = [p.item() for p in prediction]
print(prediction) # --> [1, 1, 0] (0 for Non-Offensive, 1 for Offenisve)


2026-01-20 15:29:11.210857: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768922951.547960      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768922951.658347      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768922952.554636      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768922952.554689      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768922952.554692      55 computation_placer.cc:177] computation placer alr

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

[1, 1, 0]


In [2]:
print(model_output)

(tensor([[-2.5772,  3.6957],
        [-2.8714,  3.9722],
        [ 1.9610, -3.6111]], grad_fn=<AddmmBackward0>),)


# 台湾的政治仇恨言论的检测器davidcliao

In [16]:
import requests

# URL to your Flair model file (direct link to the raw file)
model_file_url = 'https://github.com/davidycliao/taiwan-political-hatespeech-detection/raw/main/ch-hs-model/best-model.pt'

# Local path to save the model file
model_file_path = 'best-model.pt'

# Download the model
response = requests.get(model_file_url)
with open(model_file_path, 'wb') as file:
    file.write(response.content)


In [20]:
from flair.data import Sentence
from flair.models import TextClassifier
import pandas as pd

# 1. 加载模型 (你可以直接指向下载好的 best-model.pt 路径)
# 如果你没下载，通常可以尝试这个 URL（但建议下载到本地更快）
model_path = 'best-model.pt' 
classifier = TextClassifier.load(model_path)

# 2. 准备你的数据
texts = ['你就是个XX！', '黑人很多都好吃懒做。', '男女平等，黑人也很优秀。', "耶狗懂啥？"]

results = []
for text in texts:
    sentence = Sentence(text)
    # 3. 预测
    classifier.predict(sentence)
    
    # 4. 获取标签和分数
    # 假设标签是 '0' (非仇恨) 和 '1' (仇恨)
    label = sentence.labels[0].value
    score = sentence.labels[0].score
    results.append({'text': text, 'is_hate': label, 'confidence': score})

# 5. 整理成 DataFrame
df = pd.DataFrame(results)
print(df)

           text          is_hate  confidence
0       你就是个XX！  Non Hate Speech    0.561480
1    黑人很多都好吃懒做。      Hate Speech    0.788846
2  男女平等，黑人也很优秀。  Non Hate Speech    0.662776
3         耶狗懂啥？      Hate Speech    0.806530


# Morit的模型

In [24]:
from transformers import pipeline

# 1. 创建零样本分类流水线 (模型会自动处理分类头)
# 这个模型会被当作“逻辑法官”，判断你的句子是否符合“仇恨言论”这个描述
classifier = pipeline("zero-shot-classification", model="morit/chinese_xlm_xnli")

texts = ['你就是个XX！', '黑人很多都好吃懒做。', '男女平等，黑人也很优秀。', "耶狗懂啥？"]

# 2. 定义你想检测的标签名称 (可以是中文)
candidate_labels = ["仇恨言论", "正常言论", "政治敏感"]

# 3. 运行推理
for text in texts:
    result = classifier(text, candidate_labels)
    print(f"结果如下{result}")
    # 结果会按得分高低排序
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    print(f"文本: {text}")
    print(f"判定: {top_label} (置信度: {top_score:.4f})")
    print("-" * 30)

结果如下{'sequence': '你就是个XX！', 'labels': ['仇恨言论', '政治敏感', '正常言论'], 'scores': [0.42426663637161255, 0.3060622811317444, 0.26967108249664307]}
文本: 你就是个XX！
判定: 仇恨言论 (置信度: 0.4243)
------------------------------
结果如下{'sequence': '黑人很多都好吃懒做。', 'labels': ['仇恨言论', '正常言论', '政治敏感'], 'scores': [0.447601854801178, 0.3424866199493408, 0.20991159975528717]}
文本: 黑人很多都好吃懒做。
判定: 仇恨言论 (置信度: 0.4476)
------------------------------
结果如下{'sequence': '男女平等，黑人也很优秀。', 'labels': ['正常言论', '政治敏感', '仇恨言论'], 'scores': [0.7601931095123291, 0.20879894495010376, 0.03100789524614811]}
文本: 男女平等，黑人也很优秀。
判定: 正常言论 (置信度: 0.7602)
------------------------------
结果如下{'sequence': '耶狗懂啥？', 'labels': ['仇恨言论', '正常言论', '政治敏感'], 'scores': [0.43994152545928955, 0.307556688785553, 0.2525017261505127]}
文本: 耶狗懂啥？
判定: 仇恨言论 (置信度: 0.4399)
------------------------------
